## 02 — Bivariate EDA

> **Bivariate exploratory analysis of the cleaned NBA dataset.**  
> Objective: quantify relationships between variables, measure signal against the `W/L` target, detect multicollinearity, and generate feature-engineering hypotheses.  
> **Input:** `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv`  
> **Output:** correlation diagnostics, target-separated comparisons, home/away analysis, and team-level performance summaries.

---

**Pipeline**

1. Setup — imports, statistical thresholds, plotting defaults, and dataset loading
2. Correlations Between Numerical Variables — heatmaps, correlated clusters, and high-correlation pairs
3. Feature vs Target — win/loss distribution comparisons for predictive signal
4. Home vs Away Analysis — venue effects and global home-court advantage
5. Team Analysis — franchise-level net rating and win-rate variation

### 1. Setup

* **What:** Imports statistical tests, multicollinearity diagnostics, visualization tools, and shared project configuration.
* **Why:** To prepare a consistent environment for measuring relationships between features and the win/loss target.
* **Reasoning/Choices:** Pearson correlation, hypothesis tests, and VIF are included because bivariate EDA must quantify both predictive signal and redundancy among candidate features.

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

# Set the root path to the project directory
ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.outliers_influence import variance_inflation_factor

import itables.options as opt
from itables import init_notebook_mode, show

from src.loader import load_config

# Suppress warnings
warnings.filterwarnings("ignore")

# Load project configuration
config = load_config()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

# Initialize interactive notebook settings
init_notebook_mode(all_interactive=True)
opt.warn_on_undocumented_option = False
opt.maxBytes   = 0
opt.pageLength = 15
opt.lengthMenu = [15, 25, 50, 100]
opt.scrollX    = True
opt.classes    = "display nowrap compact"
opt.style      = "width:100%; font-size:13px;"

# ── Plotting configuration ────────────────────────────────────────────────────
NBA_PALETTE  = ["#1d428a", "#c8102e", "#fdb927", "#007a33", "#552583", "#ce1141"]
ACCENT       = "#1d428a"
WIN_COLOR    = "#6FB6E9"
LOSS_COLOR   = "#F47C20"
HOME_COLOR   = "#1d428a"
AWAY_COLOR   = "#fdb927"
NEUTRAL      = "#6c757d"
FIG_W_SINGLE = 10
FIG_H_SINGLE = 4
FIG_W_GRID   = 16
FIG_H_ROW    = 4
DPI          = 130

# Set seaborn theme and matplotlib parameters
sns.set_theme(style="whitegrid", palette=NBA_PALETTE)
plt.rcParams.update({
    "figure.dpi"       : DPI,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
})

# ── Statistical thresholds ───────────────────────────────────────────────────
ALPHA         = 0.05
VIF_THRESHOLD = 5.0
HIGH_CORR     = 0.80

NOTEBOOK_VERSION = "1.0.0"
log.info(f"Bivariate EDA v{NOTEBOOK_VERSION} — initialized")

#### 1.1 Dataset Loading

* **What:** Loads the cleaned NBA dataset using the configured season range and stable identifier types.
* **Why:** To ensure all relationship analysis is based on the same processed artifact used in univariate EDA.
* **Reasoning/Choices:** Keeping the load step explicit makes split size, season coverage, and team coverage visible before any statistical comparison is performed.

In [ ]:
# Set start and end years based on configuration
start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

# Format season strings for file naming
start_season    = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season      = f"{end_year}-{str(end_year + 1)[-2:]}"
DATASET_VERSION = f"{start_season}_{end_season}"

# Define the path to the cleaned dataset
PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]
CLEAN_FILE    = PROCESSED_DIR / f"cleaned_nba_data_{DATASET_VERSION}.csv"

# Load the dataset using Polars
df = pl.read_csv(
    CLEAN_FILE,
    try_parse_dates=True,
    schema_overrides={"game_id": pl.Int64, "team_id": pl.Int64},
)

# Log dataset metadata
log.info(f"Loaded  : {CLEAN_FILE.name}")
log.info(f"Shape   : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
log.info(f"Seasons : {df['season'].n_unique()}  ({df['season'].min()} – {df['season'].max()})")
log.info(f"Teams   : {df['team_abbreviation'].n_unique()}")

#### 1.2 Home/Away Splitting and Variable Definition

* **What:** Splits the dataset into home, away, win, and loss subsets while grouping numerical variables by analytical domain.
* **Why:** To support symmetric comparisons for venue effects and target separation.
* **Reasoning/Choices:** Home/away and W/L partitions are defined once to keep later plots and statistical tests consistent across performance, shooting, and contextual variables.

In [ ]:
# ── Separazione home / away per analisi simmetriche ──────────────────────────
# Separating dataset into Home and Away subsets for comparative analysis
df_home = df.filter(pl.col("home_away") == "Home")
df_away = df.filter(pl.col("home_away") == "Away")

# ── Variabili numeriche per dominio ──────────────────────────────────────────
# Defining lists of variables grouped by their analytical domain
PERF_VARS     = ["pts", "off_rating", "def_rating", "net_rating", "pace", "ts_pct"]
SHOOTING_VARS = ["fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]
ALL_NUM_VARS  = ["pts", "plus_minus", "off_rating", "def_rating", "net_rating",
                 "pace", "ts_pct", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]

# ── Target binario ───────────────────────────────────────────────────────────
# Ensuring the win/loss column is treated as a categorical/string type
df = df.with_columns(pl.col("wl").cast(pl.Utf8))
df_w = df.filter(pl.col("wl") == "W")
df_l = df.filter(pl.col("wl") == "L")

# Log findings to verify partition counts
log.info(f"Home rows : {df_home.shape[0]:,}")
log.info(f"Away rows : {df_away.shape[0]:,}")
log.info(f"Win  rows : {df_w.shape[0]:,}")
log.info(f"Loss rows : {df_l.shape[0]:,}")

### 2. Correlations Between Numerical Variables

* **What:** Measures pairwise associations among numerical basketball metrics.
* **Why:** To identify redundant predictors, tightly coupled box-score measures, and variables that may carry similar model information.
* **Reasoning/Choices:** Correlation analysis informs feature engineering by separating meaningful signal from multicollinearity that can complicate interpretation.

#### 2.1 Pearson Correlation Heatmap

* **What:** Visualizes linear correlations across all selected numerical variables.
* **Why:** To quickly locate strong positive or negative relationships and clusters of related metrics.
* **Reasoning/Choices:** Pearson correlation is appropriate here as a first-pass linear diagnostic; nonlinear relationships are interpreted cautiously and validated with additional plots or model behavior.

In [ ]:
# Calculate the Pearson correlation matrix for numerical variables
corr_matrix = (
    df.select(ALL_NUM_VARS)
    .to_pandas()
    .corr(method="pearson")
)

# Create a mask for the upper triangle to improve readability
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Visualize the correlation matrix using a heatmap
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={"size": 9},
    ax=ax,
)
ax.set_title("Pearson Correlation Matrix — all numerical variables", pad=14)
plt.tight_layout()
plt.show()

# Identify and log feature pairs with high multicollinearity
high_corr_pairs = [
    (c1, c2) for c1 in corr_matrix.columns for c2 in corr_matrix.columns
    if c1 < c2 and abs(corr_matrix.loc[c1, c2]) > HIGH_CORR
]
log.info(f"Pairs with |r| > {HIGH_CORR}: {high_corr_pairs}")

#### 2.2 Clusters of Highly Correlated Variables

* **What:** Groups variables that move together strongly across team-game observations.
* **Why:** To understand where the dataset contains overlapping information before model training.
* **Reasoning/Choices:** Highly correlated groups often reflect basketball identities, such as made shots and points, so the goal is interpretation and feature awareness rather than automatic deletion.

In [ ]:
# ── Clustermap with dendrogram ───────────────────────────────────────────────
# Decide whether to keep or remove the dendrogram
g = sns.clustermap(
    corr_matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8},
    linewidths=0.5,
    figsize=(13, 12),
    method="ward", # Uses Ward's method for hierarchical clustering
)
g.figure.suptitle(
    "Clustermap — Hierarchical clustering of variables (Ward linkage)",
    y=1.01, fontsize=13,
)
plt.show()

#### 2.3 Table of Pairs with High Correlation (|r| > 0.80)

* **What:** Lists feature pairs whose absolute Pearson correlation exceeds the high-correlation threshold.
* **Why:** To make multicollinearity explicit and reviewable before selecting modeling features.
* **Reasoning/Choices:** The 0.80 threshold flags strong redundancy while leaving room for basketball metrics that are related but still analytically distinct.

In [ ]:
# Identify pairs with high correlation
high_corr_pairs = []
for i, c1 in enumerate(corr_matrix.columns):
    for c2 in corr_matrix.columns[i + 1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) >= HIGH_CORR:
            high_corr_pairs.append({
                "var_1"       : c1,
                "var_2"       : c2,
                "pearson_r"   : round(r, 4),
                "abs_r"       : round(abs(r), 4),
                # Categorize implication based on correlation strength
                "implication" : "Redundant — consider removal/PCA" if abs(r) >= 0.95
                                else "High correlation — monitor during feature engineering",
            })

# Create DataFrame and sort by descending absolute correlation
high_corr_df = pl.DataFrame(high_corr_pairs).sort("abs_r", descending=True)
log.info(f"Pairs with |r| >= {HIGH_CORR}: {len(high_corr_pairs)}")
show(high_corr_df.to_pandas())

### 3. Feature vs Target (W / L)

* **What:** Compares numerical feature distributions between wins and losses.
* **Why:** To estimate which variables have the strongest univariate relationship with the prediction target.
* **Reasoning/Choices:** Target-separated distributions help distinguish features that merely describe game style from features that discriminate winning outcomes.

#### 3.1 Boxplots by Win/Loss Outcome

* **What:** Displays each numerical variable split by `W` and `L` outcomes.
* **Why:** To visually assess separation, overlap, and outliers for candidate predictors.
* **Reasoning/Choices:** Boxplots are compact and robust for this comparison because NBA metrics can include legitimate extreme performances that should be interpreted, not automatically discarded.

In [ ]:
# Calculate dimensions for the grid
n_vars  = len(ALL_NUM_VARS)
n_cols  = 3
n_rows  = (n_vars + n_cols - 1) // n_cols

df_pd = df.select(ALL_NUM_VARS + ["wl"]).to_pandas()

# Create a grid of boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(FIG_W_GRID, n_rows * FIG_H_ROW))
axes = axes.flatten()

# Iterate through each variable and generate a boxplot
for i, var in enumerate(ALL_NUM_VARS):
    sns.boxplot(
        data=df_pd, x="wl", y=var,
        palette={"W": WIN_COLOR, "L": LOSS_COLOR},
        order=["W", "L"],
        width=0.5,
        fliersize=2,
        ax=axes[i],
    )
    axes[i].set_title(var.upper())
    axes[i].set_xlabel("")
    axes[i].set_ylabel(var)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribution by Win/Loss — all numerical variables", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 4. Home vs Away Analysis

* **What:** Compares performance distributions and win rates between home and away teams.
* **Why:** To quantify home-court advantage and decide whether venue context should enter the modeling dataset.
* **Reasoning/Choices:** Venue effects are structural in NBA scheduling, so they should be measured directly before being converted into home-away feature differentials.

#### 4.1 Comparison of Main Distributions for Home and Away

* **What:** Compares selected performance metrics across home and away observations.
* **Why:** To identify whether venue is associated with measurable differences in scoring, efficiency, or pace.
* **Reasoning/Choices:** Distribution-level comparison is more informative than a single average because home-court effects may appear in spread, tails, or consistency.

In [ ]:
KEY_VARS = ["pts", "net_rating", "off_rating", "def_rating", "pace", "ts_pct"]
df_ha    = df.select(KEY_VARS + ["home_away"]).to_pandas()

# Calculate grid dimensions
n_cols = 3
n_rows = (len(KEY_VARS) + n_cols - 1) // n_cols

# Create a grid of histograms for Home vs Away distributions
fig, axes = plt.subplots(n_rows, n_cols, figsize=(FIG_W_GRID, n_rows * FIG_H_ROW))
axes = axes.flatten()

for i, var in enumerate(KEY_VARS):
    for label, color in [("Home", HOME_COLOR), ("Away", AWAY_COLOR)]:
        # Filter data and plot density histogram
        subset = df_ha[df_ha["home_away"] == label][var].dropna()
        axes[i].hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
        # Add a dashed line for the median
        axes[i].axvline(subset.median(), color=color, lw=1.5, ls="--")
    axes[i].set_title(var.upper())
    axes[i].set_xlabel(var)
    axes[i].set_ylabel("Density")
    axes[i].legend(fontsize=9)

# Hide any empty subplots in the grid
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Home vs Away distributions — key variables", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

#### 4.2 Global Home vs Away Win Rate

* **What:** Calculates the overall win rate for home teams compared with away teams.
* **Why:** To establish a simple, interpretable benchmark for home-court advantage.
* **Reasoning/Choices:** This rate is a baseline measure: if models cannot improve on venue-only intuition, their added feature complexity should be questioned.

In [ ]:
df_wl_ha = df.select(["home_away", "wl"]).to_pandas()

# Calculate win rate for Home vs Away
win_rate = (
    df_wl_ha
    .groupby("home_away")["wl"]
    .apply(lambda s: (s == "W").mean())
    .reset_index()
    .rename(columns={"wl": "win_rate"})
)

# Plot the win rate comparison
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    win_rate["home_away"], win_rate["win_rate"],
    color=[HOME_COLOR, AWAY_COLOR], edgecolor="white", width=0.45,
)

# Add value labels on top of the bars
for bar, val in zip(bars, win_rate["win_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
            f"{val:.1%}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# Add a reference line at 50%
ax.axhline(0.5, ls="--", color=NEUTRAL, lw=1)
ax.set_ylim(0, 0.7)
ax.set_ylabel("Win rate")
ax.set_title("Overall win rate: Home vs Away (2000–2026)")
plt.tight_layout()
plt.show()

### 5. Team Analysis

* **What:** Evaluates team-level performance variation across seasons and franchises.
* **Why:** To identify persistent strength differences that may support lagged team-quality features.
* **Reasoning/Choices:** Team effects are dynamic rather than fixed; analyzing them by season prevents historical dominance or rebuilding periods from being averaged into one misleading franchise estimate.

#### 5.1 Top and Bottom Teams by Average Net Rating per Season

* **What:** Ranks teams within seasons by average `net_rating`.
* **Why:** To compare franchise strength while controlling for era and schedule context.
* **Reasoning/Choices:** Net rating is preferred over raw points because it captures scoring margin per possession and is more stable across pace differences.

In [ ]:
# Calculate seasonal stats for each team
team_season_stats = (
    df.group_by(["team_abbreviation", "season"])
    .agg([
        pl.col("net_rating").mean().alias("net_rating_mean"),
        (pl.col("wl") == "W").mean().alias("win_rate"),
        pl.len().alias("games"),
    ])
    .sort(["season", "net_rating_mean"], descending=[False, True])
)

# Calculate overall performance metrics for each team
team_overall = (
    df.group_by("team_abbreviation")
    .agg([
        pl.col("net_rating").mean().alias("net_rating_mean"),
        (pl.col("wl") == "W").mean().alias("win_rate"),
        pl.len().alias("games"),
    ])
    .sort("net_rating_mean", descending=True)
)

# Identify top 5 and bottom 5 teams by net rating
top5    = team_overall.head(5)
bottom5 = team_overall.tail(5)
highlight = pl.concat([top5, bottom5]).to_pandas()

# Create a horizontal bar chart to compare team net ratings
fig, ax = plt.subplots(figsize=(10, 6))
colors  = [WIN_COLOR] * 5 + [LOSS_COLOR] * 5
bars    = ax.barh(
    highlight["team_abbreviation"],
    highlight["net_rating_mean"],
    color=colors, edgecolor="white",
)
ax.axvline(0, color=NEUTRAL, lw=1)

# Annotate bars with exact values
for bar, val in zip(bars, highlight["net_rating_mean"]):
    offset = 0.1 if val >= 0 else -0.1
    ha     = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", ha=ha, fontsize=9)

ax.set_xlabel("Average NET_RATING (2000–2026)")
ax.set_title("Top 5 and Bottom 5 teams by average NET_RATING (overall)")
plt.tight_layout()
plt.show()

show(team_overall.to_pandas())

#### 5.2 Win Rate Distribution by Team

* **What:** Summarizes how team win rates vary across the dataset.
* **Why:** To understand the spread of team quality and the persistence of winning behavior.
* **Reasoning/Choices:** Win rate is intuitive but schedule- and era-dependent, so it complements rather than replaces efficiency metrics such as net rating.

In [ ]:
wr_pd = team_overall.to_pandas().sort_values("win_rate", ascending=False)

# Create a bar chart for win rate per team
fig, ax = plt.subplots(figsize=(FIG_W_GRID, 5))
# Color bars based on whether the win rate is above or below 50%
bar_colors = [WIN_COLOR if w >= 0.5 else LOSS_COLOR for w in wr_pd["win_rate"]]
ax.bar(wr_pd["team_abbreviation"], wr_pd["win_rate"], color=bar_colors, edgecolor="white", width=0.7)

# Add a reference line at 50%
ax.axhline(0.5, ls="--", color=NEUTRAL, lw=1.2)
ax.set_xticklabels(wr_pd["team_abbreviation"], rotation=90, fontsize=8)
ax.set_ylabel("Win rate")
ax.set_title("Win rate per team (aggregate 2000–2026)")
ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()